In [1]:
! pip install sentence_transformers umap_learn hdbscan

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 94.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 75.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 42.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 1.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 29.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.3 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 8.1 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 78.5 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.

In [2]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# Access the secret
user_secrets =  UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

# login in to Hugging Face

login(token=hf_token)

In [3]:
import pandas as pd

file_path = '/kaggle/input/final-dataset/dataset_with_key_phrases (3).csv'

try:
    df = pd.read_csv(file_path, sep=',', encoding='cp1252', on_bad_lines='warn')

    print("Dataset loaded successfully! Here's a sample:")
    display(df.head())

    df_filtered = df.copy()

    # Clean the 'SOurce' column before filtering
    # .str.strip() removes leading/trailing whitespace
    # .str.lower() converts all text to lowercase
    df_filtered['source'] = df_filtered['source'].str.strip().str.lower()

    final_df = df_filtered[df_filtered['source'] == 'rentlingo']
    print(f"\nSuccessfully filtered for 'RentLingo'. Found {len(final_df)} matching rows.")

    if len(final_df) == 0:
        print("Warning: The Final_df is empty. Check if 'RentLingo' exists in the 'source' column.")
    else:
        display(final_df.head())
except FileNotFoundError:
    print(f"File not found at: {file_path}")
    print("Please double-check the path in the Kaggle 'Data' tab.")

except Exception as e:
    print(f"AN error occurred: {e}")


Dataset loaded successfully! Here's a sample:


,category,title,body,amenities,source,key_phrases
0,housing/rent/apartment,"Studio apartment 2nd St NE, Uhland Terrace NE,...","This unit is located at second St NE, Uhland T...",NaN,RentLingo,"['located at second St NE, Uhland Terrace NE, ..."
1,housing/rent/apartment,Studio apartment 814 Schutte Road,"This unit is located at 814 Schutte Road, Evan...",NaN,RentLingo,"['814 Schutte Road, Evansville, 47712, IN', 'M..."
2,housing/rent/apartment,"Studio apartment N Scott St, 14th St N, Arling...","This unit is located at N Scott St, 14th St N,...",NaN,RentLingo,"['N Scott St, 14th St N, Arlington, VA 22209',..."
3,housing/rent/apartment,Studio apartment 1717 12th Ave,"This unit is located at 1717 12th Ave, Seattle...",NaN,RentLingo,"['1717 12th Ave, Seattle, 98122', 'Monthly ren..."
4,housing/rent/apartment,"Studio apartment Washington Blvd, N Cleveland ...","This unit is located at Washington Blvd, N Cle...",NaN,RentLingo,"['Washington Blvd, N Cleveland St, Arlington, ..."



Successfully filtered for 'RentLingo'. Found 6912 matching rows.


,category,title,body,amenities,source,key_phrases
0,housing/rent/apartment,"Studio apartment 2nd St NE, Uhland Terrace NE,...","This unit is located at second St NE, Uhland T...",NaN,rentlingo,"['located at second St NE, Uhland Terrace NE, ..."
1,housing/rent/apartment,Studio apartment 814 Schutte Road,"This unit is located at 814 Schutte Road, Evan...",NaN,rentlingo,"['814 Schutte Road, Evansville, 47712, IN', 'M..."
2,housing/rent/apartment,"Studio apartment N Scott St, 14th St N, Arling...","This unit is located at N Scott St, 14th St N,...",NaN,rentlingo,"['N Scott St, 14th St N, Arlington, VA 22209',..."
3,housing/rent/apartment,Studio apartment 1717 12th Ave,"This unit is located at 1717 12th Ave, Seattle...",NaN,rentlingo,"['1717 12th Ave, Seattle, 98122', 'Monthly ren..."
4,housing/rent/apartment,"Studio apartment Washington Blvd, N Cleveland ...","This unit is located at Washington Blvd, N Cle...",NaN,rentlingo,"['Washington Blvd, N Cleveland St, Arlington, ..."


In [4]:
import ast 

# Convert the column from string to list

# This function will try to convert a value to a list. If it fails,
# it returns an empty list to avoid errors.

def safe_literal_eval(val):
    try:
        # Check if the value is a string and looks like a list
        if isinstance(val, str)  and val.strip().startswith('['):
            return ast.literal_eval(val)
        # If it's already a list, just return it
        elif isinstance(val, list):
            return val

    except (ValueError, SyntaxError):
        # If parsing fails, return an empty list
        pass
    return [] # Return empty list for NANs or malformed strings

final_df['key_phrases_list'] = final_df['key_phrases'].apply(safe_literal_eval)

all_phrases = final_df['key_phrases_list'].explode().dropna().unique().tolist()

print(f"Found {len(all_phrases)} unique key phrases.")
print("Here's a sample:")
print(all_phrases[:10])

Found 13034 unique key phrases.
Here's a sample:
['located at second St NE, Uhland Terrace NE, Washington, DC 20002', 'monthly rental rates range from $790 - $1090', 'studio units available for rent', '814 Schutte Road, Evansville, 47712, IN', 'Monthly rental rates range from $425 - $445', 'studio - 1 beds units available for rent', 'N Scott St, 14th St N, Arlington, VA 22209', 'Monthly rental rates range from $1390', 'Studio units available for rent', '1717 12th Ave, Seattle, 98122']


/tmp/ipykernel_36/138010858.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  final_df['key_phrases_list'] = final_df['key_phrases'].apply(safe_literal_eval)


In [6]:
from sentence_transformers import SentenceTransformer

phrases_for_clustering = [f"clustering_document: {phrase}" for phrase in all_phrases]
print("Added 'clustering_document:' prefix to all phrases.")

model = SentenceTransformer("google/embeddinggemma-300m")

print("Generating embeddings... This may take a few minutes.")
embeddings = model.encode(
    phrases_for_clustering,
    show_progress_bar=True,
    normalize_embeddings=True
)

print(f"Embeddings created with shape: {embeddings.shape}")

Added 'clustering_document:' prefix to all phrases.


modules.json:   0%|          | 0.00/573 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/997 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/58.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

3_Dense/model.safetensors:   0%|          | 0.00/9.44M [00:00<?, ?B/s]

Generating embeddings... This may take a few minutes.


Batches:   0%|          | 0/408 [00:00<?, ?it/s]

Embeddings created with shape: (13034, 768)


In [8]:
import umap

reducer = umap.UMAP(n_neighbors=15, n_components=10, min_dist=0.0, metric='cosine')
embedding_10d = reducer.fit_transform(embeddings)

print(f"Dimensionality reduced to shape: {embedding_10d.shape}")

Dimensionality reduced to shape: (13034, 10)


In [9]:
import hdbscan

clusterer = hdbscan.HDBSCAN(min_cluster_size = 200, min_samples=50, metric='euclidean')
cluster_labels = clusterer.fit_predict(embedding_10d)

df_clusters = pd.DataFrame({
    'phrase' : all_phrases,
    'cluster': cluster_labels,
})

n_clusters = len(df_clusters['cluster'].unique()) - 1 # -1 for noise cluster
print(f"HDBSCAN found {n_clusters} clusters.")

print("\nCluster membership counts:")
print(df_clusters['cluster'].value_counts())

HDBSCAN found 25 clusters.

Cluster membership counts:
cluster
 1     1093
 20    1020
 0      775
 23     738
 21     721
 19     647
 12     595
 11     574
 18     570
 16     551
 4      551
 8      530
 6      528
 24     519
 14     421
 13     406
 9      337
 17     333
 2      318
 5      313
 7      296
 3      276
 10     272
 15     270
 22     234
-1      146
Name: count, dtype: int64


In [11]:
import numpy as np

unique_clusters = np.unique(cluster_labels)
unique_clusters = unique_clusters[unique_clusters != -1]

cluster_samples_data =[]

print(f"\n--- Collecting samples from {len(unique_clusters)} clusters ---")

for cluster_id in unique_clusters:
    phrases_in_cluster = df_clusters[df_clusters['cluster'] == cluster_id]['phrase'].tolist()

    sample_size = min(len(phrases_in_cluster), 10)

    for phrase in phrases_in_cluster[:sample_size]:
        cluster_samples_data.append({
            'cluster_id': cluster_id,
            'sample_phrase': phrase
        })

df_samples = pd.DataFrame(cluster_samples_data)

output_path = '/kaggle/working/cluster_samples.csv'
df_samples.to_csv(output_path, index=False)

print(f"\n CLuster samples successfully saved to: {output_path}")


--- Collecting samples from 25 clusters ---

 CLuster samples successfully saved to: /kaggle/working/cluster_samples.csv


In [12]:
output_path = '/kaggle/working/clusters.csv'
df_clusters.to_csv(output_path, index=False)

print(f"\n Clusters successfully saved to: {output_path}")


 Clusters successfully saved to: /kaggle/working/clusters.csv
